In [1]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import (
    Adam,
    SGD,
    RMSprop
)
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)
import keras_tuner as kt
import pandas as pd

In [2]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [3]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df["label"] = encoder.fit_transform(train_df["disease"])
valid_df["label"] = encoder.transform(valid_df["disease"])
test_df["label"] = encoder.transform(test_df["disease"])

In [4]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 32

In [5]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [6]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [7]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [8]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [9]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [11]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [12]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [13]:
cnn_dropout = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dropout(0.5),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
cnn_dropout.compile(
    optimizer="SGD",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
history_dropout = cnn_dropout.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 204s 913ms/step - accuracy: 0.1789 - loss: 1.9411 - val_accuracy: 0.5080 - val_loss: 1.7886 - learning_rate: 0.0100
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 199s 892ms/step - accuracy: 0.3738 - loss: 1.8864 - val_accuracy: 0.5200 - val_loss: 1.4833 - learning_rate: 0.0100
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 205s 921ms/step - accuracy: 0.4049 - loss: 1.7840 - val_accuracy: 0.0439 - val_loss: 2.2459 - learning_rate: 0.0100
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 587s 3s/step - accuracy: 0.3949 - loss: 1.7315 - val_accuracy: 0.4108 - val_loss: 1.8243 - learning_rate: 0.0100
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 166s 746ms/step - accuracy: 0.4372 - loss: 1.6634 - val_accuracy: 0.4148 - val_loss: 1.3515 - learning_rate: 0.0100
Restoring model weights from the end of the best epoch: 5.


In [16]:
train_loss, lr_train_acc_drop = cnn_dropout.evaluate(train_ds)
valid_loss, lr_valid_acc_drop = cnn_dropout.evaluate(valid_ds)
test_loss, lr_test_acc_drop = cnn_dropout.evaluate(test_ds)
print(lr_train_acc_drop)
print(lr_valid_acc_drop)
print(lr_test_acc_drop)

220/220 ━━━━━━━━━━━━━━━━━━━━ 53s 230ms/step - accuracy: 0.4257 - loss: 1.3024
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 231ms/step - accuracy: 0.4168 - loss: 1.3478
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 231ms/step - accuracy: 0.3999 - loss: 1.3632
0.4256775975227356
0.416777640581131
0.3998669385910034


In [17]:
drop_results = pd.DataFrame(columns=[
    "Model",
    "Train accuracy",
    "Test accuracy",
    "Valid accuracy"
])
drop_results.loc[len(drop_results)] = [
    "dropout cnn using SGD",
    lr_train_acc_drop,
    lr_test_acc_drop,
    lr_valid_acc_drop
]
drop_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,dropout cnn using SGD,0.425678,0.399867,0.416778


In [18]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [19]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [20]:
cnn_dropout = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dropout(0.5),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
cnn_dropout.compile(
    optimizer="RMSprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [22]:
history_dropout = cnn_dropout.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 202s 904ms/step - accuracy: 0.3882 - loss: 1.9748 - val_accuracy: 0.5712 - val_loss: 1.1845 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 210s 944ms/step - accuracy: 0.4213 - loss: 1.7077 - val_accuracy: 0.3995 - val_loss: 1.5206 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 217s 971ms/step - accuracy: 0.4496 - loss: 1.6183 - val_accuracy: 0.5879 - val_loss: 1.0603 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 210s 941ms/step - accuracy: 0.4539 - loss: 1.6165 - val_accuracy: 0.4774 - val_loss: 1.2520 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 201s 899ms/step - accuracy: 0.4471 - loss: 1.5736 - val_accuracy: 0.1804 - val_loss: 1.6570 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 3.


In [23]:
train_loss, rm_train_acc_drop = cnn_dropout.evaluate(train_ds)
valid_loss, rm_valid_acc_drop = cnn_dropout.evaluate(valid_ds)
test_loss, rm_test_acc_drop = cnn_dropout.evaluate(test_ds)
print(rm_train_acc_drop)
print(rm_valid_acc_drop)
print(rm_test_acc_drop)

220/220 ━━━━━━━━━━━━━━━━━━━━ 53s 229ms/step - accuracy: 0.5996 - loss: 1.0265
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 238ms/step - accuracy: 0.5712 - loss: 1.0706
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 236ms/step - accuracy: 0.5875 - loss: 1.0686
0.5995720624923706
0.5712383389472961
0.5874916911125183


In [24]:
drop_results.loc[len(drop_results)] = [
    "dropout cnn using RMSprop",
    rm_train_acc_drop,
    rm_test_acc_drop,
    rm_valid_acc_drop
]
drop_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,dropout cnn using SGD,0.425678,0.399867,0.416778
1,dropout cnn using RMSprop,0.599572,0.587492,0.571238


In [25]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [26]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [27]:
cnn_dropout = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dropout(0.5),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
cnn_dropout.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [29]:
history_dropout = cnn_dropout.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 816ms/step - accuracy: 0.3509 - loss: 1.7843 - val_accuracy: 0.0413 - val_loss: 2.1855 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 108s 481ms/step - accuracy: 0.3585 - loss: 1.6705 - val_accuracy: 0.2570 - val_loss: 1.8096 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 151s 681ms/step - accuracy: 0.4203 - loss: 1.5522 - val_accuracy: 0.4261 - val_loss: 1.5985 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 191s 861ms/step - accuracy: 0.4558 - loss: 1.6254 - val_accuracy: 0.3908 - val_loss: 1.5334 - learning_rate: 0.0010
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 188s 844ms/step - accuracy: 0.4305 - loss: 1.5569 - val_accuracy: 0.4720 - val_loss: 1.2696 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 5.


In [30]:
train_loss, adam_train_acc_drop = cnn_dropout.evaluate(train_ds)
valid_loss, adam_valid_acc_drop = cnn_dropout.evaluate(valid_ds)
test_loss, adam_test_acc_drop = cnn_dropout.evaluate(test_ds)
print(adam_train_acc_drop)
print(adam_valid_acc_drop)
print(adam_test_acc_drop)

220/220 ━━━━━━━━━━━━━━━━━━━━ 49s 213ms/step - accuracy: 0.4692 - loss: 1.2404
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 212ms/step - accuracy: 0.4687 - loss: 1.2565
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 218ms/step - accuracy: 0.4531 - loss: 1.2802
0.4691868722438812
0.4687083959579468
0.45309382677078247


In [31]:
drop_results.loc[len(drop_results)] = [
    "dropout cnn using adam",
    adam_train_acc_drop,
    adam_test_acc_drop,
    adam_valid_acc_drop
]
drop_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,dropout cnn using SGD,0.425678,0.399867,0.416778
1,dropout cnn using RMSprop,0.599572,0.587492,0.571238
2,dropout cnn using adam,0.469187,0.453094,0.468708


In [32]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 64

In [33]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [34]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [35]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [36]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [37]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [38]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [39]:
cnn_dropout = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dropout(0.5),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [40]:
cnn_dropout.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [41]:
batch_size = 64
history_dropout = cnn_dropout.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
    batch_size = batch_size
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.3826 - loss: 1.9451 - val_accuracy: 0.1152 - val_loss: 1.9514
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 153s 1s/step - accuracy: 0.3372 - loss: 1.8470 - val_accuracy: 0.4181 - val_loss: 1.6286
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 185s 2s/step - accuracy: 0.4427 - loss: 1.7497 - val_accuracy: 0.4987 - val_loss: 1.4253
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - accuracy: 0.4417 - loss: 1.5766 - val_accuracy: 0.6005 - val_loss: 1.1710
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 186s 2s/step - accuracy: 0.4772 - loss: 1.4604 - val_accuracy: 0.2417 - val_loss: 1.8205


In [42]:
train_loss, bt64_train_acc_drop = cnn_dropout.evaluate(train_ds)
valid_loss, bt64_valid_acc_drop = cnn_dropout.evaluate(valid_ds)
test_loss, bt64_test_acc_drop = cnn_dropout.evaluate(test_ds)
print(bt64_train_acc_drop)
print(bt64_valid_acc_drop)
print(bt64_test_acc_drop)

110/110 ━━━━━━━━━━━━━━━━━━━━ 48s 416ms/step - accuracy: 0.2455 - loss: 1.7910
24/24 ━━━━━━━━━━━━━━━━━━━━ 10s 398ms/step - accuracy: 0.2430 - loss: 1.8187
24/24 ━━━━━━━━━━━━━━━━━━━━ 10s 400ms/step - accuracy: 0.2322 - loss: 1.8296
0.2455064207315445
0.2430093139410019
0.23220226168632507


In [43]:
drop_results.loc[len(drop_results)] = [
    "dropout cnn using batchsize 64",
    bt64_train_acc_drop,
    bt64_test_acc_drop,
    bt64_valid_acc_drop
]
drop_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,dropout cnn using SGD,0.425678,0.399867,0.416778
1,dropout cnn using RMSprop,0.599572,0.587492,0.571238
2,dropout cnn using adam,0.469187,0.453094,0.468708
3,dropout cnn using batchsize 64,0.245506,0.232202,0.243009


In [44]:
def build_dropout_cnn(hp):
    model=tf.keras.Sequential([
        Conv2D(
            32,
            (3,3),
            activation="relu",
            input_shape=INPUT_SHAPE
        ),
        MaxPooling2D(),
        Flatten(),
        Dense(
            128,
            activation="relu"
        ),
        Dropout(
            hp.Float(
                "dropout",
                0.2,
                0.6,
                0.1
            )
        ),
        Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])
    model.compile(
        optimizer=hp.Choice(
            "optimizer",
            ["adam","rmsprop"]
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [45]:
dropout_tuner = kt.RandomSearch(
    build_dropout_cnn,
    objective="val_accuracy",
    max_trials=3,
    directory="tuning",
    project_name="cnn_dropout"
)
dropout_tuner.search(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights
)

Trial 3 Complete [00h 05m 47s]
val_accuracy: 0.44074568152427673

Best val_accuracy So Far: 0.540612518787384
Total elapsed time: 00h 17m 34s


In [46]:
best_drop_cnn = dropout_tuner.get_best_models(1)[0]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [47]:
best_hps = dropout_tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'dropout': 0.5, 'optimizer': 'adam'}


In [48]:
cnn_dropout = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dropout(0.5),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [49]:
cnn_dropout.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [50]:
history_dropout = cnn_dropout.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5
)

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 163s 1s/step - accuracy: 0.6536 - loss: 1.1796 - val_accuracy: 0.6678 - val_loss: 0.9360
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 167s 1s/step - accuracy: 0.6688 - loss: 0.9440 - val_accuracy: 0.6698 - val_loss: 0.9190
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 174s 2s/step - accuracy: 0.6729 - loss: 0.8942 - val_accuracy: 0.6804 - val_loss: 0.8617
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 169s 2s/step - accuracy: 0.6785 - loss: 0.8661 - val_accuracy: 0.6824 - val_loss: 0.8412
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 166s 1s/step - accuracy: 0.6845 - loss: 0.8546 - val_accuracy: 0.6871 - val_loss: 0.8594


In [51]:
train_loss, hype_train_acc_drop = cnn_dropout.evaluate(train_ds)
valid_loss, hype_valid_acc_drop = cnn_dropout.evaluate(valid_ds)
test_loss, hype_test_acc_drop = cnn_dropout.evaluate(test_ds)
print(hype_train_acc_drop)
print(hype_valid_acc_drop)
print(hype_test_acc_drop)

110/110 ━━━━━━━━━━━━━━━━━━━━ 46s 395ms/step - accuracy: 0.6909 - loss: 0.8152
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 371ms/step - accuracy: 0.6911 - loss: 0.8563
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 372ms/step - accuracy: 0.6800 - loss: 0.8611
0.6908701658248901
0.6910785436630249
0.6799733638763428


In [52]:
drop_results.loc[len(drop_results)] = [
    "dropout cnn hyperparameter",
    hype_train_acc_drop,
    hype_test_acc_drop,
    hype_valid_acc_drop
]
drop_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,dropout cnn using SGD,0.425678,0.399867,0.416778
1,dropout cnn using RMSprop,0.599572,0.587492,0.571238
2,dropout cnn using adam,0.469187,0.453094,0.468708
3,dropout cnn using batchsize 64,0.245506,0.232202,0.243009
4,dropout cnn hyperparameter,0.690870,0.679973,0.691079


In [53]:
drop_results.to_csv("drop_cnn_comparison.csv",index=False)

In [54]:
best_drop_cnn.save("cnn_dropout_phase5.keras")

In [55]:
cnn_dropout.save("cnn_dropout_bestmodel.keras")